In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Data extraction
import kagglehub

import os
import glob
import ast
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Timing
import time

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

from sklearn.metrics import (
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier

# Dimensionality Reduction
from sklearn.decomposition import PCA

# Clustering
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

# Imbalanced Learning
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.under_sampling import ClusterCentroids
from imblearn.combine import SMOTETomek

# Feature Extraction
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
import shap

# Data manipulation
from sklearn.model_selection import StratifiedShuffleSplit

import sys

# Misc
import warnings
warnings.filterwarnings("ignore")

# Plot style
sns.set_theme(style="whitegrid")


# Results Analysis 

This section analyzes the results produced in **Q1–Q3**.

The goal is to identify the best-performing configurations before running the remaining experiments (Q4–Q5),

As well as examine the feature-importance results.

The analysis is performed in a few parts:
- **(1) Model Analysis**
- **(2) Feature Analysis**
- **(3) Random Forest's SHAP interaction analysis** - driven from the conclusions of (1) and (2)

---

## Load Transaction Data

In [2]:
# Download latest version if it doesn't exist yet
dataset_key = "mlg-ulb/creditcardfraud"
csv_name = "creditcardfraud.csv"

if not os.path.exists(csv_name):
    path = kagglehub.dataset_download(dataset_key)
else:
    path = csv_name

csv_files = glob.glob(path + "/**/*.csv", recursive=True)
print(csv_files)
df = pd.read_csv(csv_files[0])

target_col = "Class"

y = df[target_col].values
X = df.drop(columns=[target_col]).values

feature_names = df.drop(columns=[target_col]).columns.tolist()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", len(feature_names))

['/home/noam/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3/creditcard.csv']
X shape: (284807, 30)
y shape: (284807,)
Num features: 30


# Models Results: (1)

We examine the following configuration parameters:
- Models
- Data balancing methods
- Class weights
- Recall optimization

## Load Experiment Results

Load the evaluation results and feature analysis generated during the Q1–Q3 experiments.

In [ ]:
current_path = Path.cwd()

while current_path.name != "FinalProject":
    current_path = current_path.parent


PROJECT_ROOT = current_path


results_df = pd.read_csv(
    PROJECT_ROOT / "out" / "Q1_3_output.csv",
    keep_default_na=False
)

sys.path.append(str(PROJECT_ROOT))

results_df.head()

print(f"Results: {len(results_df)} rows")
results_df.head()

## Dataset Overview

Inspect the structure of the evaluation results before performing any analysis.

In [ ]:
results_df.info()

results_df.describe()

## Best Configuration Analysis

We intend to observe the reuslts and determine which model performed best, and which configuration should we run it with.

The configuration being:
- Data Balancing Method
- Class weight
- Recall Threshold

### Examine Best Performance

We determine the best performance by sorting for Recall, after filtering for Precision >= 0.5 .

Reasoning:
- We're trying to optimize recall while maintaining a reasonable precision
- We noticed that once precision goes below 50% it usually sinks towards the 0~10%

**Conclusion**: Random Forest performed best

In [ ]:
best_model_configs = (
    results_df[
        (results_df["precision"] > 0.5)
    ]
    .sort_values("recall", ascending=False)
    .groupby("model")
    .first()
)

best_model_configs[
    [
        "balancing",
        "class_weight",
        "threshold_mode",
        "precision",
        "recall",
        "f1",
        "pr_auc",
        "roc_auc"
    ]
]

## Analyzing Our Recall Demand

In fraud detection, missing fraudulent transactions (false negatives) is usually more costly than investigating additional legitimate transactions.

Therefore, we also tried to achieve at least 95% recall, by influencing the model's decision using a costum threshold.

Here we'll examine the configurations that stood up to our expectations, and observe their precision.

**Conclusion**: Our threshold was too high, sinking precisions. Therefore we should not use it from now on.

**Note**: Ideally we would run a lower threshold and try to optimize for 0.9 recall, but due to lack of time we won't do that for all models.

In [ ]:
high_recall_models = (
    results_df[
        results_df["recall"] >= 0.95
    ]
    .sort_values(
        ["precision", "pr_auc"],
        ascending=False
    )
    .groupby("model")
    .first()
)

high_recall_models[
    [
        "balancing",
        "class_weight",
        "threshold_mode",
        "precision",
        "recall",
        "f1",
        "pr_auc",
        "roc_auc"
    ]
]

## Effect of Balancing Methods

Class imbalance is a major challenge in fraud detection because fraudulent transactions represent only a small portion of the dataset.

Different balancing techniques are evaluated to determine whether they improve minority-class detection.

The methods compared are:

- **None**: Original imbalanced dataset.
- **SMOTE**: Synthetic generation of minority-class samples.
- **SMOTE-Tomek**: SMOTE combined with Tomek link removal to reduce overlapping samples.
- **Undersampling**: Reduction of majority-class samples.
- **Cluster Centroids**: Majority-class reduction using clustering.

For this comparison, we aggregate results across all models and configurations to observe the general effect of each balancing approach.

## Overall Results   

We can note that overall the balancing did not improve the results, and rather harm them sometimes.

Mostly, the balancing made models improve recall while sinking precision, rendering the balance useless.

Therefore we'll **discuss below** only the entries where both precision and recall are above 0.5 

In [ ]:
balancing_effect = results_df[
    (results_df["class_weight"] == "None") &
    (results_df["threshold_mode"] == "default")
]

balancing_effect = balancing_effect[
    [
        "model",
        "balancing",
        "class_weight",
        "precision",
        "recall",
        "f1",
        "pr_auc",
        "roc_auc"
    ]
]

balancing_effect.sort_values(
    ["model", "recall", "precision"],
    ascending=[True, False, False]
)

## Balancing Results Conclusion

Filtering by precision and recall over 0.5 shows us that:
- The only benifactor of the balancing methods is **Random Forest**

The addition of data balancing resulted in Random Forest having a better recall,

while sacrificing a bit of it's precision.

Given the sort by recall, we can see that SMOTE_Tomek helped us increase recall to 85% + while maintaining precision above 80%.

**Conclusion**: We'll use Random Forest with SMOTE_Tomek


In [ ]:
balancing_effect = results_df[
    (results_df["threshold_mode"] == "default")&
    (results_df["precision"] >= 0.5)&
    (results_df["recall"] >= 0.5)
]

balancing_effect = balancing_effect[
    [
        "model",
        "balancing",
        "class_weight",
        "precision",
        "recall",
        "f1",
        "pr_auc",
        "roc_auc"
    ]
]

balancing_effect.sort_values(
    ["model", "recall", "precision"],
    ascending=[True, False, False]
)

## Effect of Class Weighting

Class weighting modifies the training loss so that mistakes on minority-class samples receive a higher penalty.

Two approaches are compared:

- **None**: All samples contribute equally.
- **Balanced**: The classifier automatically assigns higher importance to fraudulent transactions.
- Note: KNN and Adaboost do not support "class_weight"

The goal is to determine whether weighting the minority class improves fraud detection performance.


-------

We can infer that the class weight was ineffective, as it did not improve the results towards the direction we desired.

For Logistic Regression:
- Improved recall
- Sank Precision (rendering the model useless)

For Random Forest:
- Decreased recall
- Increased precision

**Note**: Class weighting changes the ranking and probability distribution, not necessarily the final classification threshold. Which explains the results.

**Conclusion**: We will not use class_weight feature anymore during our runs


In [ ]:
class_weight_effect = results_df[
    (results_df["balancing"] == "None") &
    (results_df["threshold_mode"] == "default") &
    (results_df["model"].isin(["Logistic Regression", "Random Forest"]))
]

class_weight_effect = class_weight_effect[
    [
        "model",
        "class_weight",
        "precision",
        "recall",
        "f1",
        "pr_auc",
        "roc_auc"
    ]
]

class_weight_effect.sort_values(
    ["model", "recall"],
    ascending=[True, False]
)

# Final Model-Configuration Conclusions:

1. The best performing model was **Random Forest**, using **SMOTE_Tomek** balancing method
2. **Class weight** and **Recall optimization** turned out to not improve our results
3. Given we only tried to optimized for recall >= 95 , it might be worth to **optimize for a lower recall** to improve results 

----

# Feature Importance Analysis (2)

After evaluating model performance, we analyze feature importance to understand which features contribute most to fraud detection.

We examine three types of importance:

1. **Dataset-level importance**
   - Pearson correlation
   - Spearman correlation
   - Mutual information
   
   These methods measure the relationship between each feature and the target independently of the model.

2. **Model-based importance**
   - Feature importance extracted from the trained model
   - SHAP values

   These methods explain which features influenced the model's predictions.

In [ ]:
current_path = Path.cwd()

while current_path.name != "FinalProject":
    current_path = current_path.parent


PROJECT_ROOT = current_path


features_df = pd.read_csv(
    PROJECT_ROOT / "out" / "Q1_3_features.csv",
    keep_default_na=False
)

print(f"Feature logs: {len(features_df)} rows")

features_df.head()

In [ ]:
features_df[
    [
        "model",
        "balancing",
        "class_weight"
    ]
].drop_duplicates()

features_df.columns

### Parsing Feature Dictionaries

The feature importance values were stored as strings inside the CSV.
We convert them back into dictionaries so they can be analyzed.

In [ ]:
dict_columns = [
    "model_importance_top",
    "shap_importance_top",
    "pearson_top",
    "spearman_top",
    "mutual_info_top"
]

for col in dict_columns:
    features_df[col] = features_df[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    
features_df.head()

## Dataset-Level Feature Analysis

The following analysis identifies features that show the strongest relationship with the target variable.

- Pearson correlation captures linear relationships.
- Spearman correlation captures monotonic relationships.
- Mutual information captures nonlinear dependencies.

In [ ]:
dataset_features = features_df.iloc[0]

pearson = dataset_features["pearson_top"]
spearman = dataset_features["spearman_top"]
mutual_info = dataset_features["mutual_info_top"]

In [ ]:
def plot_feature_dict(values, title):
    values = dict(sorted(values.items(), key=lambda x: abs(x[1]), reverse=True))

    plt.figure(figsize=(10,4))
    plt.bar(
        [str(k) for k in values.keys()],
        values.values()
    )
    plt.xticks(rotation=45)
    plt.title(title)
    plt.ylabel("Score")
    plt.show()


plot_feature_dict(
    pearson,
    "Top Pearson Correlation Features"
)

plot_feature_dict(
    spearman,
    "Top Spearman Correlation Features"
)

plot_feature_dict(
    mutual_info,
    "Top Mutual Information Features"
)

| Method | What it measures | Value Range | Interpretation |
|---|---|---|---|
| Pearson Correlation | Linear relationship between a feature and the target | 0.01 - 0.15 | Some features show weak linear association with fraud, but no single feature strongly separates fraud from non-fraud cases |
| Spearman Correlation | Monotonic relationship between a feature and the target | 0.005 - 0.065 | Features have limited ordered relationships with fraud probability |
| Mutual Information | General dependency, including nonlinear relationships | 0.005 - 0.008 | Individual features provide limited information gain; fraud prediction likely depends on feature combinations |

### Data-Level Feature Analysis Conclusion

The low correlation and dependency scores across all methods indicate that no individual feature has a strong linear or monotonic relationship with fraud transactions. This suggests that fraud detection likely depends on complex feature interactions rather than a small subset of highly correlated features.

## Model-Level Feature Analysis

While correlation-based analysis evaluates features independently, model-level analysis examines which features contribute most to the predictions made by trained models. This allows us to capture feature interactions and understand the decision-making process of the selected models.

### **Logistic Regression** Feature Analysis

Logistic Regression assigns a coefficient to each feature, representing its influence on the prediction. Positive coefficients increase the likelihood of a transaction being classified as fraud, while negative coefficients decrease it.

In [ ]:
lr_features = features_df[
    (features_df["model"] == "Logistic Regression") &
    (features_df["balancing"] == "None") &
    (features_df["class_weight"] == "None")
]

lr_features[
    [
        "model",
        "balancing",
        "class_weight",
        "model_importance_top",
        "shap_importance_top"
    ]
]

lr_model_importance = lr_features.iloc[0]["model_importance_top"]

lr_coef_df = pd.DataFrame({
    "feature": list(lr_model_importance["feature"].values()),
    "coefficient": list(lr_model_importance["importance"].values())
})

# Rank by absolute coefficient magnitude
lr_coef_df["abs_coefficient"] = lr_coef_df["coefficient"].abs()

lr_coef_df = lr_coef_df.sort_values(
    "abs_coefficient",
    ascending=False
)

lr_coef_df

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(
    lr_coef_df["feature"],
    lr_coef_df["coefficient"]
)

plt.xticks(rotation=45)
plt.ylabel("Coefficient magnitude")
plt.title("Logistic Regression Feature Importance")

plt.show()

### SHAP Feature Importance

SHAP analysis identifies the features with the largest contribution to Logistic Regression predictions.  
The most influential features were `Time`, `V4`, and `V3`, indicating their higher impact on model decisions.

Model coefficients represent the direct contribution of each feature to the Logistic Regression decision boundary, while SHAP values explain the contribution of each feature to individual predictions.  
SHAP was used to complement coefficient analysis by providing a more interpretable view of feature impact on the model's outputs.

In [ ]:
lr_shap_importance = lr_features.iloc[0]["shap_importance_top"]

lr_shap_df = pd.DataFrame({
    "feature": list(lr_shap_importance["feature"].values()),
    "shap_importance": list(lr_shap_importance["shap_importance"].values())
})

lr_shap_df = lr_shap_df.sort_values(
    "shap_importance",
    ascending=False
)

lr_shap_df

In [ ]:
# Plot SHAP feature importance
plt.figure(figsize=(10, 5))

plt.bar(
    lr_shap_df["feature"],
    lr_shap_df["shap_importance"]
)

plt.xlabel("Feature")
plt.ylabel("Mean |SHAP value|")
plt.title("Logistic Regression SHAP Feature Importance")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

## Conclusions:

Logistic Regression coefficients identified V25, V15, V14, and V22 as the features with the strongest global influence on the model's decision boundary. SHAP analysis complemented this by showing which features contributed most to actual predictions, highlighting Time, V4, V3, and V15 as important drivers of model decisions.

----

## **Random Forest** Feature Analysis

Random Forest evaluates feature importance based on how much each feature contributes to improving the model's decisions across multiple decision trees. Unlike Logistic Regression coefficients, feature importance in Random Forest does not represent a direct positive or negative relationship with fraud, but rather the overall contribution of each feature to the model's predictive performance.

### Raw Random Forest Analysis
Below we'll extract Random Forest with no changes to it's loss or any changes to the data

In [ ]:
rf_features = features_df[
    (features_df["model"] == "Random Forest") &
    (features_df["balancing"] == "None") &
    (features_df["class_weight"] == "None")
]

rf_features[
    [
        "model",
        "balancing",
        "class_weight",
        "model_importance_top",
        "shap_importance_top"
    ]
]

rf_model_importance = rf_features.iloc[0]["model_importance_top"]

rf_importance_df = pd.DataFrame({
    "feature": list(rf_model_importance["feature"].values()),
    "importance": list(rf_model_importance["importance"].values())
})

rf_importance_df = rf_importance_df.sort_values(
    "importance",
    ascending=False
)

rf_importance_df

### Smote_Tomek Random Forest Analysis
Below we'll extract Random Forest while Smote_Tomek balancing method was applied

In [ ]:
rf_smote_tomek_features = features_df[
    (features_df["model"] == "Random Forest") &
    (features_df["balancing"] == "SMOTE_Tomek") &
    (features_df["class_weight"] == "None")
]

rf_smote_tomek_features[
    [
        "model",
        "balancing",
        "class_weight",
        "model_importance_top",
        "shap_importance_top"
    ]
]

rf_smote_tomek_model_importance = rf_smote_tomek_features.iloc[0]["model_importance_top"]

rf_smote_tomek_importance_df = pd.DataFrame({
    "feature": list(rf_smote_tomek_model_importance["feature"].values()),
    "importance": list(rf_smote_tomek_model_importance["importance"].values())
})

rf_smote_tomek_importance_df = rf_smote_tomek_importance_df.sort_values(
    "importance",
    ascending=False
)

rf_smote_tomek_importance_df

In [ ]:
# Add configuration labels
rf_importance_df_plot = rf_importance_df.copy()
rf_importance_df_plot["configuration"] = "RF Baseline"

rf_smote_tomek_importance_df_plot = rf_smote_tomek_importance_df.copy()
rf_smote_tomek_importance_df_plot["configuration"] = "RF SMOTE-Tomek"


# Combine
rf_comparison_df = pd.concat(
    [
        rf_importance_df_plot,
        rf_smote_tomek_importance_df_plot
    ],
    ignore_index=True
)


# Order features by SMOTE-Tomek importance
feature_order = (
    rf_smote_tomek_importance_df
    .sort_values(
        "importance",
        ascending=False
    )["feature"]
    .tolist()
)


plt.figure(figsize=(12, 5))

sns.barplot(
    data=rf_comparison_df,
    x="feature",
    y="importance",
    hue="configuration",
    hue_order=[
        "RF SMOTE-Tomek",
        "RF Baseline"
    ],
    order=feature_order,
    palette=[
        "tab:red",  # RF SMOTE-Tomek
        "tab:blue"     # RF Baseline
    ]
)


plt.xlabel("Feature")
plt.ylabel("Feature Importance")
plt.title(
    "Random Forest Feature Importance Comparison"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()

plt.show()

## Conclusions:

Random Forest identified V17, V14, and V12 as the most important features based on their contribution to tree-based decisions. Since the importance is distributed across multiple features rather than concentrated in a few variables, the model likely relies on complex feature interactions when making predictions.

**Note**: We lost Random Forest's SHAP values during the run. We MAY run it again specifically on Random Forest later, though that data is missing for now.

---



# Final Conclusions Feature Analysis (2)

1. The **correlation analysis** indicates that there are **no strong direct relationships** between individual features and the transaction class, suggesting that fraud detection depends on more complex patterns rather than single-feature effects.

2. **Logistic Regression coefficient analysis** shows that features **V25, V15, V14, and V22** have the strongest influence on the model's global decision boundary.

3. **Logistic Regression SHAP analysis** provides local explanations of predictions and highlights that **Time** has the largest contribution to individual predictions, followed by **V4 and V3**.

4. **Random Forest feature importance** shows that features such as **V17, V14, and V12** contribute most to tree decisions. However, the distributed importance across multiple features suggests that Random Forest likely captures complex feature interactions rather than relying on a small number of dominant predictors.

**Note:** Given that Random Forest is our best performing model, it would be interesting to run SHAP's TreeExplainer on it specifically as well as observe how these results may vary using different data-balancing methods

---

# Shap Interaction Analysis (3)

In this section we'll act on the note we discussed, and analyize pairs of features rather than singulars. We'll discuss their affect on Random Forest - our best model.

### Random Forest - SHAP; Feature and Interaction Analysis

This experiment compares a baseline Random Forest with a SMOTE-Tomek balanced Random Forest to analyze how feature interactions influence model decisions. SHAP TreeExplainer interaction values are used to identify important feature pairs that contribute jointly to fraud classification.

Below we defined an helper func `compute_rf_shap_stats`  to compute the SHAP values and the experiment func ` run_rf_interaction_experiment`

In [ ]:
# Import the function to compute SHAP interaction values
from src.shap_interaction import compute_rf_shap_stats


In [ ]:
def run_rf_interaction_experiment(
    X,
    y,
    feature_names,
    explain_samples=1000,
    top_k=10
):

    results = {}

    # ============================================
    # Global Train/Test Split
    # ============================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )


    # ============================================
    # SHAP Explanation Sampling
    # Take fraud samples from test set
    # Fill remaining samples with normal transactions
    # ============================================

    X_test_df = pd.DataFrame(
        X_test,
        columns=feature_names
    )

    y_test_series = pd.Series(
        y_test
    )


    fraud_mask = (
        y_test_series == 1
    )

    X_fraud = X_test_df[fraud_mask]

    fraud_count = len(X_fraud)


    normal_count = (
        explain_samples - fraud_count
    )


    # If requested samples exceed available fraud cases,
    # use all fraud cases and fill remaining with normal cases

    X_normal = (
        X_test_df[~fraud_mask]
        .sample(
            normal_count,
            random_state=42
        )
    )


    X_explain = pd.concat(
        [
            X_fraud,
            X_normal
        ]
    )


    y_explain = pd.concat(
        [
            y_test_series[fraud_mask],
            y_test_series[X_test_df.index.isin(X_normal.index)]
        ]
    )


    X_explain = X_explain.reset_index(
        drop=True
    )

    y_explain = y_explain.reset_index(
        drop=True
    )


    print("\nSHAP explanation set:")
    print(
        f"Fraud samples: {sum(y_explain == 1)}"
    )
    print(
        f"Normal samples: {sum(y_explain == 0)}"
    )
    print(
        f"Total explanation samples: {len(X_explain)}"
    )


    configurations = {

        "RF_Baseline": None,

        "RF_SMOTE_Tomek": SMOTETomek(
            random_state=42
        )

    }


    # ============================================
    # Run Configurations
    # ============================================

    for config_name, balancer in configurations.items():

        print("\n================================")
        print(
            f"Running configuration: {config_name}"
        )
        print("================================")


        # -----------------------------
        # Apply balancing
        # -----------------------------

        if balancer is not None:

            print(
                "Applying SMOTE_Tomek..."
            )

            X_train_bal, y_train_bal = (
                balancer.fit_resample(
                    X_train,
                    y_train
                )
            )

        else:

            X_train_bal = X_train
            y_train_bal = y_train


        print(
            f"Training samples: {len(X_train_bal)}"
        )


        # -----------------------------
        # Train RF
        # -----------------------------

        rf = RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )


        print(
            "Training Random Forest..."
        )


        rf.fit(
            X_train_bal,
            y_train_bal
        )


        print(
            "Random Forest trained."
        )


        print(
            f"Explaining {len(X_explain)} samples with SHAP."
        )


        # -----------------------------
        # SHAP + interactions
        # -----------------------------

        shap_stats = compute_rf_shap_stats(
            rf,
            X_explain,
            y_explain,
            feature_names,
            config_name,
            top_k
        )


        results[config_name] = shap_stats


    print(
        "\nFinished RF interaction experiment."
    )


    # ============================================
    # Save SHAP Results
    # ============================================

    os.makedirs("out", exist_ok=True)


    # -------------------------
    # SHAP feature importance
    # -------------------------

    shap_rows = []


    for config, result in results.items():

        shap_df = (
            result["shap_importance"]
            .reset_index()
        )

        shap_df.columns = [
            "feature",
            "shap_importance"
        ]

        shap_df.insert(
            0,
            "configuration",
            config
        )

        shap_rows.append(
            shap_df
        )


    shap_results_df = pd.concat(
        shap_rows,
        ignore_index=True
    )


    shap_results_df.to_csv(
        "out/rf_shap_importance.csv",
        index=False
    )


    # -------------------------
    # SHAP interaction importance
    # -------------------------

    interaction_rows = []


    interaction_types = [

        "interaction_global_mean",
        "interaction_global_median",

        "interaction_fraud_mean",
        "interaction_fraud_median",

        "interaction_normal_mean",
        "interaction_normal_median"

    ]


    for config, result in results.items():

        for interaction_type in interaction_types:

            interaction_df = (
                result[interaction_type]
                .copy()
            )


            interaction_df.insert(
                0,
                "configuration",
                config
            )


            interaction_df.insert(
                1,
                "interaction_type",
                interaction_type
            )


            interaction_rows.append(
                interaction_df
            )


    interaction_results_df = pd.concat(
        interaction_rows,
        ignore_index=True
    )


    interaction_results_df.to_csv(
        "out/rf_shap_interactions.csv",
        index=False
    )


    print("Saved:")
    print("- out/rf_shap_importance.csv")
    print("- out/rf_shap_interactions.csv")


    return results

In [ ]:
"""
# Run Random Forest SHAP Interaction Experiment

rf_interaction_results = run_rf_interaction_experiment(
    X,
    y,
    feature_names,
    explain_samples=1000,
    top_k=10
)
rf_interaction_results["RF_Baseline"].keys()
"""

## Load SHAP Analysis Results

The following analysis compares Random Forest explanations under two configurations:
- Baseline Random Forest
- Random Forest with SMOTE-Tomek balancing

SHAP feature importance is used to analyze individual feature contributions, while SHAP interaction values are used to identify feature combinations influencing predictions.

In [ ]:
shap_df = pd.read_csv(
    PROJECT_ROOT / "out" / "rf_shap_importance.csv"
)

interaction_df = pd.read_csv(
    PROJECT_ROOT / "out" / "rf_shap_interactions.csv"
)

print(f"SHAP Importance: {len(shap_df)} rows")
print(shap_df.head())
print("===============================")
print(f"SHAP Interactions: {len(interaction_df)} rows")
print(interaction_df.head())

## Individual Feature Importance

SHAP values measure the average contribution of each feature to the model's predictions. Higher values indicate features that have a stronger influence on the model output.

In [ ]:
# Order features by RF_SMOTE_Tomek SHAP importance

feature_order = (
    shap_df[
        shap_df["configuration"] == "RF_SMOTE_Tomek"
    ]
    .sort_values(
        "shap_importance",
        ascending=False
    )["feature"]
    .tolist()
)


plt.figure(figsize=(10,5))

sns.barplot(
    data=shap_df,
    x="feature",
    y="shap_importance",
    hue="configuration",
    order=feature_order
)

plt.title(
    "Random Forest SHAP Feature Importance"
)

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

### Interpetation

SMOTE-Tomek increased the magnitude of SHAP feature contributions, indicating that balancing allowed Random Forest to learn stronger fraud-related patterns. Features V14, V4, and V12 became the most influential after balancing.

---

## Global SHAP Interaction Importance

Global interaction values measure how much feature combinations contribute to the model prediction across all analyzed transactions.

In [ ]:
# Select global mean interactions
global_interactions = interaction_df[
    interaction_df["interaction_type"] == "interaction_global_mean"
].copy()


# Create feature pair labels
global_interactions["feature_pair"] = (
    global_interactions["feature_1"]
    + "-"
    + global_interactions["feature_2"]
)


# Order pairs by RF_SMOTE_Tomek interaction values
interaction_order = (
    global_interactions[
        global_interactions["configuration"] == "RF_SMOTE_Tomek"
    ]
    .sort_values(
        "interaction",
        ascending=False
    )["feature_pair"]
    .tolist()
)


plt.figure(figsize=(12,5))

sns.barplot(
    data=global_interactions,
    x="feature_pair",
    y="interaction",
    hue="configuration",
    order=interaction_order
)


plt.title(
    "Global SHAP Feature Interactions"
)

plt.xlabel(
    "Feature Pair"
)

plt.ylabel(
    "Mean Absolute SHAP Interaction"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.show()

## Fraud-Specific SHAP Interactions

To analyze patterns specific to fraudulent transactions, interaction values were calculated only on fraud samples from the test set.

In [ ]:
fraud_interactions = interaction_df[
    interaction_df["interaction_type"]
    ==
    "interaction_fraud_mean"
]


fraud_interactions.head(20)

In [ ]:
# Create feature pair labels
fraud_interactions = fraud_interactions.copy()

fraud_interactions["feature_pair"] = (
    fraud_interactions["feature_1"]
    + "-"
    + fraud_interactions["feature_2"]
)


# Order by RF_SMOTE_Tomek fraud interaction strength

fraud_interaction_order = (
    fraud_interactions[
        fraud_interactions["configuration"] == "RF_SMOTE_Tomek"
    ]
    .sort_values(
        "interaction",
        ascending=False
    )["feature_pair"]
    .tolist()
)


plt.figure(figsize=(12,6))


sns.barplot(
    data=fraud_interactions,
    x="feature_pair",
    y="interaction",
    hue="configuration",
    order=fraud_interaction_order
)


plt.title(
    "Fraud-Specific SHAP Feature Interactions"
)

plt.xlabel(
    "Feature Pair"
)

plt.ylabel(
    "Mean Absolute SHAP Interaction"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.show()

## Conclusion:

It seems that the baseline Random Forest managed to 

### Compare Random Forest Explanations

The following results summarize the three most influential features and feature interactions identified by each explanation method for both the baseline Random Forest model and the Random Forest trained using SMOTE-Tomek balancing.

In [ ]:
# ==========================================
# Random Forest Top-3 Feature Analysis
# ==========================================

def print_top3(series, name):
    print(f"\n{name}")
    print("-" * len(name))
    print(series.head(3).to_string())


# -------------------------
# Built-in Feature Importance
# -------------------------

print_top3(
    rf_importance_df.set_index("feature")["importance"],
    "RF Baseline - Built-in Feature Importance"
)

print_top3(
    rf_smote_tomek_importance_df.set_index("feature")["importance"],
    "RF SMOTE-Tomek - Built-in Feature Importance"
)


# -------------------------
# SHAP Feature Importance
# -------------------------

shap_baseline = shap_df[
    shap_df["configuration"] == "RF_Baseline"
].set_index("feature")["shap_importance"].sort_values(
    ascending=False
)

shap_smote = shap_df[
    shap_df["configuration"] == "RF_SMOTE_Tomek"
].set_index("feature")["shap_importance"].sort_values(
    ascending=False
)


print_top3(
    shap_baseline,
    "RF Baseline - SHAP Feature Importance"
)

print_top3(
    shap_smote,
    "RF SMOTE-Tomek - SHAP Feature Importance"
)


# -------------------------
# SHAP Global Interactions
# -------------------------

global_interactions = interaction_df[
    interaction_df["interaction_type"] == "interaction_global_mean"
].copy()

global_interactions["feature_pair"] = (
    global_interactions["feature_1"]
    + "-"
    + global_interactions["feature_2"]
)


global_baseline = global_interactions[
    global_interactions["configuration"] == "RF_Baseline"
].set_index("feature_pair")["interaction"].sort_values(
    ascending=False
)


global_smote = global_interactions[
    global_interactions["configuration"] == "RF_SMOTE_Tomek"
].set_index("feature_pair")["interaction"].sort_values(
    ascending=False
)


print_top3(
    global_baseline,
    "RF Baseline - SHAP Global Interactions"
)

print_top3(
    global_smote,
    "RF SMOTE-Tomek - SHAP Global Interactions"
)


# -------------------------
# SHAP Fraud-Specific Interactions
# -------------------------

fraud_interactions = interaction_df[
    interaction_df["interaction_type"] == "interaction_fraud_mean"
].copy()

fraud_interactions["feature_pair"] = (
    fraud_interactions["feature_1"]
    + "-"
    + fraud_interactions["feature_2"]
)


fraud_baseline = fraud_interactions[
    fraud_interactions["configuration"] == "RF_Baseline"
].set_index("feature_pair")["interaction"].sort_values(
    ascending=False
)


fraud_smote = fraud_interactions[
    fraud_interactions["configuration"] == "RF_SMOTE_Tomek"
].set_index("feature_pair")["interaction"].sort_values(
    ascending=False
)


print_top3(
    fraud_baseline,
    "RF Baseline - SHAP Fraud-Specific Interactions"
)

print_top3(
    fraud_smote,
    "RF SMOTE-Tomek - SHAP Fraud-Specific Interactions"
)

## Comparison Table For RF

Below we'll compare the feature importance of Random Forest (raw and smote-tomek enabled) using the built-in method, Shap, and Shap-interactions.

For readability, we assigned an emoji-colour for each variable to emphasize repeats in feature appearance:

- 🔴 V14
- 🔵 V17
- 🟢 V12
- 🟠 V4
- 🟣 V10



| Configuration      | Method                               | Top-3 Features / Interactions         | Interpretation                                                                                                                                                                                                  |
| ------------------ | ------------------------------------ | ------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **RF Baseline**    | **Built-in Feature Importance**      | 🔵V17, 🔴V14, 🟢V12                   | The model primarily relies on these features when constructing decision trees. Feature importance reflects how much each feature reduces impurity during training, but does not explain individual predictions. |
| **RF Baseline**    | **SHAP Feature Importance**          | 🔵V17, 🔴V14, 🟢V12                   | SHAP confirms the same three features as the most influential, indicating that the baseline model's global behavior is largely driven by these individual variables.                                            |
| **RF Baseline**    | **SHAP Global Interactions**         | 🔴V14-🔵V17, 🟢V12-🔵V17, 🟢V12-🔴V14 | The strongest global interactions occur between the same features that are individually important, suggesting the model captures limited but meaningful relationships among them across all transactions.       |
| **RF Baseline**    | **SHAP Fraud-Specific Interactions** | 🔴V14-🔵V17, 🟢V12-🔵V17, 🟢V12-🔴V14 | The same interactions become substantially stronger when considering only fraudulent transactions, indicating these feature combinations are particularly characteristic of fraud detection.                    |
| **RF SMOTE-Tomek** | **Built-in Feature Importance**      | 🔴V14, 🟣V10, 🟠V4                      | After balancing, the Random Forest changes its decision strategy and relies more heavily on features associated with the minority (fraud) class. Shifting the importance to two new features (🟣,🟠)                                                                |
| **RF SMOTE-Tomek** | **SHAP Feature Importance**          | 🔴V14, 🟠V4, 🟢V12                    | SHAP shows that balancing increases the influence of several fraud-related features, with 🔴V14 remaining the dominant contributor while 🟠V4 becomes significantly more important.                             |
| **RF SMOTE-Tomek** | **SHAP Global Interactions**         | 🟠V4-🔴V14, 🟢V12-🔴V14, 🟠V4-🟢V12   | SMOTE-Tomek leads the model to learn stronger interactions between important features, indicating a greater reliance on combinations of variables rather than isolated features.                                |
| **RF SMOTE-Tomek** | **SHAP Fraud-Specific Interactions** | 🔴V14-🔵V17, 🟠V4-🔵V17, 🟣V10-🔵V17    | Fraud-specific interactions differ from the global interactions, suggesting that the balanced model identifies fraudulent transactions using distinct combinations of features centered around 🔵V17.           |


---

# Final Conclusions (3)

1. **🔴V14** consistently appeared as the most influential feature across nearly all Random Forest analyses, indicating a major role in fraud detection.

2. The baseline Random Forest relied primarily on the individual features **🔵V17**, **🔴V14**, and **🟢V12**, with relatively weak feature interactions.

3. Applying **SMOTE-Tomek** shifted the model toward **🔴V14**, **🟠V4**, and **🟢V12**, while also increasing the importance of feature interactions.

4. Fraud-specific SHAP interactions revealed stronger and different feature combinations than the global analysis, suggesting that fraudulent transactions are identified through distinct relationships between features.

5. Overall, balancing the training data enabled the Random Forest to learn more complex fraud-related patterns rather than relying mainly on individual feature importance.